In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell1 %% [code]
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}，请检查是否在右侧 Add Data 挂载了对应的依赖库！")

# 100% 照抄原代码：寻找并安装 onnxruntime

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
else:
    print('没有！')



# 100% 照抄原代码：寻找并降级/升级 TF 2.20 以解决死锁
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
    print("✅ TF 2.20 installed from wheel")
except Exception as e:
    print(e)

# 导入所有后续需要的包
import onnxruntime as ort
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
import re

print("环境配置严格执行完毕！")

ONNX Runtime installed
✅ TF 2.20 installed from wheel
环境配置严格执行完毕！


In [3]:
# %% [code]
# cell2: 路径配置与模型加载
import onnxruntime as ort
import pandas as pd
import numpy as np
from pathlib import Path
import re

BASE = Path("/kaggle/input/competitions/birdclef-2026")
TRAIN_AUDIO_DIR = BASE / "train_audio"
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
INPUT_ROOT = Path("/kaggle/input")

# ========== 新增的 Soundscapes 路径 ==========
SOUNDSCAPES_DIR = BASE / "train_soundscapes"
SOUNDSCAPES_CSV = BASE / "train_soundscapes_labels.csv"

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = int(SR * WINDOW_SEC)  # 160000

# 1. 严格照抄原代码的 ONNX 载入方式
def find_onnx():
    for p in INPUT_ROOT.rglob("perch_v2.onnx"):
        return p
    raise FileNotFoundError("找不到 perch_v2.onnx")

ONNX_PERCH_PATH = find_onnx()

_so = ort.SessionOptions()
_so.intra_op_num_threads = 4  # 原代码指定的线程数

# 如果用 GPU 加速提特征，自动检测并启用 CUDA
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"]

ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so, providers=providers)
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print(f"✅ ONNX 模型加载成功！使用的后端: {ONNX_SESSION.get_providers()[0]}")

# 2. 严格照抄原代码的标签映射 (Genus proxy logits)
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))

NO_LABEL = len(bc_labels)
mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
label2bc = mapping.set_index("primary_label")["bc_index"].to_dict()

# 照抄作者找同属代理的代码
proxy_map = {}
for _, row in mapping[mapping["bc_index"] == NO_LABEL].iterrows():
    target = row["primary_label"]
    sci = str(row["scientific_name"])
    genus = sci.split()[0]
    hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
    if len(hits) > 0:
        proxy_map[target] = hits["bc_index"].astype(int).tolist()

def get_target_bc_indices(primary_label):
    bc_idx = label2bc.get(primary_label, NO_LABEL)
    if bc_idx != NO_LABEL:
        return [bc_idx]
    elif primary_label in proxy_map:
        return proxy_map[primary_label]
    else:
        return []

print("✅ 映射表严格建立完成。")

✅ ONNX 模型加载成功！使用的后端: CPUExecutionProvider
✅ 映射表严格建立完成。


In [ ]:
# %% [code]
# cell3: 基于官方 CSV 的精准打分与多标签拆分
import librosa
from tqdm.auto import tqdm
import gc

# 1. 加载官方标签 CSV
labels_df = pd.read_csv(SOUNDSCAPES_CSV)

# 时间格式转换辅助函数: 将 HH:MM:SS 转为秒数
def time_str_to_seconds(time_str):
    if isinstance(time_str, (int, float)):
        return float(time_str)
    h, m, s = str(time_str).split(':')
    return int(h) * 3600 + int(m) * 60 + float(s)

labels_df['start_sec'] = labels_df['start'].apply(time_str_to_seconds)
labels_df['end_sec'] = labels_df['end'].apply(time_str_to_seconds)

# 自动推断包含 primary 的列名 (根据你的截图，防止官方列名存在变体)
label_col = [c for c in labels_df.columns if 'primary' in c][0]

results = []

# 按文件分组处理，防止一个1min音频被重复读取12次
grouped = labels_df.groupby('filename')

for filename, group in tqdm(grouped, desc="Processing Soundscapes"):
    filepath = SOUNDSCAPES_DIR / filename
    
    if not filepath.exists():
        print(f"⚠️ 文件不存在，跳过: {filepath}")
        continue

    try:
        # 完整读取该音频 (默认长度应为1分钟)
        y, _ = librosa.load(str(filepath), sr=SR, mono=True)
    except Exception as e:
        print(f"读取音频失败 {filename}: {e}")
        continue

    slices_info = []
    frames = []

    # 遍历该文件所有标注的 5 秒切片
    for _, row in group.iterrows():
        start_sec = row['start_sec']
        end_sec = row['end_sec']
        
        start_idx = int(start_sec * SR)
        end_idx = int(end_sec * SR)
        
        y_slice = y[start_idx:end_idx]

        # 确保截取的刚好是标准的 5s 长度，如果不满足就 padding 或者截断
        if len(y_slice) < WINDOW_SAMPLES:
            y_slice = np.pad(y_slice, (0, WINDOW_SAMPLES - len(y_slice)))
        elif len(y_slice) > WINDOW_SAMPLES:
            y_slice = y_slice[:WINDOW_SAMPLES]

        frames.append(y_slice)
        slices_info.append(row)

    if not frames:
        continue

    # 把该文件的所有 5s 切片合成一个 batch 一次性推理
    frames = np.array(frames).astype(np.float32)
    try:
        outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: frames})
        logits = outs[ONNX_OUT_MAP["label"]] # 维度: [num_slices, 10000+类]
    except Exception as e:
        print(f"推理失败 {filename}: {e}")
        continue

    # 将推理结果按照切片和拆分后的标签对号入座
    for i, row in enumerate(slices_info):
        # 获取用分号分割的原始字符串，如 "22961;23158;24321"
        raw_labels_str = str(row[label_col])
        # 按分号拆分，去除可能的多余空格
        labels_list = [l.strip() for l in raw_labels_str.split(';') if l.strip()]

        for single_label in labels_list:
            target_indices = get_target_bc_indices(single_label)

            if len(target_indices) > 0:
                # 获取该特定物种（含 proxy）在该切片上的最高得分
                score = float(logits[i, target_indices].max())
            else:
                score = -999.0

            results.append({
                "filename": filename,
                "start": row['start'],         # 保持官方的 HH:MM:SS 格式
                "end": row['end'],             # 保持官方的 HH:MM:SS 格式
                "original_labels": raw_labels_str, # 留存参考原始的多物种字符串
                "primary_label": single_label,     # 拆解后的具体某一个物种
                "perch_score": score
            })

    # 清理内存
    gc.collect()

# 2. 导出新 CSV 格式
results_df = pd.DataFrame(results)
output_path = "/kaggle/working/train_soundscapes_perch_scores.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ 处理完成！已保存 {len(results_df)} 行数据至 {output_path}")
print("前5行预览：")
display(results_df.head())

Processing Soundscapes:   0%|          | 0/66 [00:00<?, ?it/s]